[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap08_machine_learning/exemplo_regressao_preditiva_prazo_entrega.ipynb)

# Regressão preditiva para o prazo realizado de entrega

Este notebook utiliza a Base didática Caso Tech para construir modelos de regressão que estimam o prazo realizado de entrega. O fluxo inclui definição do alvo, seleção de atributos disponíveis antes da entrega, separação dos dados, comparação de modelos e avaliação em dados de teste.

> O caminho do botão pressupõe que este notebook seja salvo no repositório no local indicado pelo URL. Se a pasta definida para o capítulo 8 for diferente, ajuste o endereço do botão.

## 1. Objetivo e enquadramento

O problema é de **aprendizado supervisionado**, pois cada registro usado no treinamento contém atributos de entrada e um prazo realizado conhecido. Como a variável-alvo é numérica e contínua, trata-se de uma tarefa de **regressão**. O objetivo didático é estimar o prazo de entrega a partir de informações que estejam disponíveis no momento em que a previsão seria feita.

A unidade de análise deve ser um pedido. A base Caso Tech possui linhas repetidas por pedido na planilha de vendas; por isso, não se deve unir diretamente as linhas sem verificar a granularidade. A preparação abaixo procura localizar automaticamente as planilhas e colunas relevantes e interrompe a execução se não conseguir identificar com segurança o prazo-alvo ou uma chave de pedido. Confira os nomes e as regras exibidos antes de interpretar os resultados.

In [ ]:
import io
import re
import unicodedata
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMENTE = 2026
URL_BASE = "https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/dados/base%20did%C3%A1tica%20Caso%20Tech.xlsx"

np.random.seed(SEMENTE)
print("Semente aleatória:", SEMENTE)

## 2. Carregamento da base

O notebook tenta carregar a base pelo endereço do repositório. Se o download não estiver disponível, permite selecionar um arquivo Excel local. A aba escolhida e a dimensão da tabela serão exibidas para conferência.

In [ ]:
def carregar_excel():
    try:
        with urllib.request.urlopen(URL_BASE, timeout=30) as resposta:
            conteudo = resposta.read()
        arquivo = pd.ExcelFile(io.BytesIO(conteudo))
        origem = "repositório GitHub"
    except Exception as erro_download:
        print("Download automático indisponível:", erro_download)
        try:
            from google.colab import files
            enviados = files.upload()
            if not enviados:
                raise RuntimeError("Nenhum arquivo foi selecionado.")
            nome = next(iter(enviados))
            arquivo = pd.ExcelFile(io.BytesIO(enviados[nome]))
            origem = nome
        except ImportError:
            raise RuntimeError("Execute no Colab ou disponibilize a planilha no ambiente.")
    return arquivo, origem

arquivo_excel, origem = carregar_excel()
print("Origem:", origem)
print("Abas disponíveis:", arquivo_excel.sheet_names)

# Exibe dimensões e colunas por aba para documentar a estrutura real.
abas = {}
for nome_aba in arquivo_excel.sheet_names:
    tabela = pd.read_excel(arquivo_excel, sheet_name=nome_aba)
    abas[nome_aba] = tabela
    print(f"\n{nome_aba}: {tabela.shape[0]} linhas x {tabela.shape[1]} colunas")
    print(list(tabela.columns))

## 3. Identificação da granularidade e do alvo

A identificação das colunas usa palavras-chave e normalização de acentos. É uma etapa auxiliar, não substitui a conferência do dicionário de dados. A variável-alvo deve representar o prazo **realizado**, por exemplo, dias entre a data do pedido e a data de entrega. Registros sem entrega realizada não são usados para treinar esse alvo.

In [ ]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", " ", texto.lower()).strip()

def localizar_coluna(colunas, grupos_palavras):
    pares = [(coluna, normalizar(coluna)) for coluna in colunas]
    for palavras in grupos_palavras:
        for original, nome in pares:
            if all(palavra in nome for palavra in palavras):
                return original
    return None

# Seleciona a aba com mais indícios de conter o prazo realizado e uma chave de pedido.
chaves_possiveis = [["pedido"], ["order", "id"], ["id", "pedido"]]
alvos_possiveis = [["prazo", "realizado"], ["dias", "entrega"], ["tempo", "entrega"], ["entrega", "real"]]
selecionada = None
for nome_aba, tabela in abas.items():
    chave = localizar_coluna(tabela.columns, chaves_possiveis)
    alvo = localizar_coluna(tabela.columns, alvos_possiveis)
    if chave is not None and alvo is not None:
        selecionada = (nome_aba, tabela, chave, alvo)
        break

if selecionada is None:
    print("\nNão foi possível identificar automaticamente as colunas necessárias.")
    print("Revise as colunas listadas acima e ajuste os grupos de palavras-chave na célula seguinte.")
else:
    nome_aba, dados_brutos, coluna_pedido, coluna_alvo = selecionada
    print("Aba selecionada:", nome_aba)
    print("Chave do pedido:", coluna_pedido)
    print("Alvo candidato:", coluna_alvo)
    display(dados_brutos[[coluna_pedido, coluna_alvo]].head())

### Conferência necessária

Se a identificação automática não encontrar a aba ou as colunas, examine a saída anterior e defina explicitamente `nome_aba`, `coluna_pedido` e `coluna_alvo` na célula abaixo. Não escolha uma variável apenas por semelhança de nome: confirme no material do caso como o prazo foi calculado e em que unidade está expresso.

In [ ]:
# Se a detecção automática funcionou, esta célula mantém os nomes encontrados.
# Caso contrário, substitua os valores None pelos nomes exatos exibidos na saída.
if "selecionada" in globals() and selecionada is not None:
    pass
else:
    nome_aba = None       # Exemplo: "Entregas"
    coluna_pedido = None  # Exemplo: "ID Pedido"
    coluna_alvo = None    # Exemplo: "Prazo realizado (dias)"
    if not all([nome_aba, coluna_pedido, coluna_alvo]):
        raise ValueError("Defina os nomes exatos da aba, da chave e do prazo realizado.")
    dados_brutos = abas[nome_aba]

if coluna_pedido not in dados_brutos.columns or coluna_alvo not in dados_brutos.columns:
    raise KeyError("A chave ou a coluna-alvo não está presente na aba selecionada.")

## 4. Preparação dos dados e prevenção de vazamento

A previsão deve utilizar somente variáveis disponíveis antes da entrega. Datas ou campos que revelem a data de entrega, o atraso observado ou qualquer informação calculada depois da conclusão do transporte não podem ser usados como preditores. A regra de deduplicação depende da estrutura das planilhas: o exemplo agrega linhas por pedido usando o primeiro valor não ausente de cada coluna, estratégia adequada apenas quando as colunas do pedido são consistentes entre as linhas. A célula compara a consistência e informa eventuais divergências; elas devem ser investigadas, não ocultadas.

In [ ]:
# Converte o alvo para número e remove pedidos sem prazo realizado.
dados = dados_brutos.copy()
dados[coluna_alvo] = pd.to_numeric(dados[coluna_alvo], errors="coerce")
dados = dados.dropna(subset=[coluna_pedido, coluna_alvo])

# Diagnóstico de duplicidade da chave.
contagem_linhas = dados.groupby(coluna_pedido, dropna=False).size()
print("Pedidos com mais de uma linha:", int((contagem_linhas > 1).sum()))
print("Maior número de linhas por pedido:", int(contagem_linhas.max()))

# Avalia se há mais de um valor não ausente por pedido nas demais colunas.
colunas_inconsistentes = []
for coluna in dados.columns:
    if coluna == coluna_pedido:
        continue
    n_valores = dados.groupby(coluna_pedido)[coluna].nunique(dropna=True)
    if (n_valores > 1).any():
        colunas_inconsistentes.append(coluna)
print("Colunas com valores divergentes dentro do pedido:", colunas_inconsistentes)

# Reduz para uma linha por pedido. O uso de first é documentado e deve ser
# validado após a inspeção das divergências acima.
dados_pedido = dados.groupby(coluna_pedido, as_index=False).first()
print("Pedidos válidos após consolidação:", len(dados_pedido))
print(dados_pedido[coluna_alvo].describe())

### Seleção dos atributos

A seleção abaixo exclui a chave, o alvo e colunas cujo nome indica informação posterior à entrega ou derivada diretamente do prazo/atraso. Também remove identificadores quase únicos. As colunas restantes são apresentadas para revisão. Para uma aplicação real, a disponibilidade temporal de cada variável precisa ser confirmada com o processo operacional.

In [ ]:
termos_vazamento = [
    "realizado", "data entrega", "entrega real", "prazo realizado",
    "atraso", "dias atraso", "data recebimento", "recebimento"
]

candidatas = []
for coluna in dados_pedido.columns:
    nome = normalizar(coluna)
    if coluna in [coluna_pedido, coluna_alvo]:
        continue
    if any(termo in nome for termo in termos_vazamento):
        continue
    # Remove identificadores quase únicos, exceto a chave que já foi retirada.
    proporcao_unicos = dados_pedido[coluna].nunique(dropna=True) / max(len(dados_pedido), 1)
    if proporcao_unicos > 0.95:
        continue
    candidatas.append(coluna)

print("Atributos candidatos:")
print(candidatas)
if not candidatas:
    raise ValueError("Nenhum atributo candidato foi identificado. Revise a base e as regras de seleção.")

## 5. Separação entre treino e teste

A divisão aleatória em treino e teste permite avaliar o desempenho em registros não usados no ajuste. A semente fixa torna a divisão reproduzível. Para este exemplo didático, utiliza-se uma divisão aleatória; em um cenário com evolução temporal ou implantação futura, a validação deve respeitar a ordem do tempo ou separar grupos de pedidos relacionados, quando aplicável.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = dados_pedido[candidatas].copy()
y = dados_pedido[coluna_alvo].astype(float)

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.20, random_state=SEMENTE
)

colunas_numericas = X_treino.select_dtypes(include=np.number).columns.tolist()
colunas_categoricas = [coluna for coluna in X_treino.columns if coluna not in colunas_numericas]

transformador_numerico = Pipeline([
    ("imputacao", SimpleImputer(strategy="median")),
    ("padronizacao", StandardScaler())
])
transformador_categorico = Pipeline([
    ("imputacao", SimpleImputer(strategy="most_frequent")),
    ("codificacao", OneHotEncoder(handle_unknown="ignore"))
])

preprocessador = ColumnTransformer([
    ("numericas", transformador_numerico, colunas_numericas),
    ("categoricas", transformador_categorico, colunas_categoricas)
], remainder="drop")

print("Treino:", X_treino.shape, "Teste:", X_teste.shape)
print("Atributos numéricos:", colunas_numericas)
print("Atributos categóricos:", colunas_categoricas)

## 6. Modelos de regressão

Comparam-se quatro referências: um preditor ingênuo baseado na mediana do treino, regressão linear, árvore de decisão e floresta aleatória. A comparação introduz famílias de algoritmos com diferentes capacidades de representar relações. O objetivo não é declarar um algoritmo universalmente superior, mas avaliar os modelos no mesmo conjunto de teste e com as mesmas métricas.

In [ ]:
modelos = {
    "Referência: mediana": DummyRegressor(strategy="median"),
    "Regressão linear": LinearRegression(),
    "Árvore de decisão": DecisionTreeRegressor(
        max_depth=5, min_samples_leaf=10, random_state=SEMENTE
    ),
    "Floresta aleatória": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=5, random_state=SEMENTE, n_jobs=-1
    )
}

resultados = []
predicoes = {}

for nome, modelo in modelos.items():
    pipeline = Pipeline([
        ("preprocessamento", preprocessador),
        ("modelo", modelo)
    ])
    pipeline.fit(X_treino, y_treino)
    previsao = pipeline.predict(X_teste)
    predicoes[nome] = previsao
    resultados.append({
        "Modelo": nome,
        "MAE": mean_absolute_error(y_teste, previsao),
        "RMSE": mean_squared_error(y_teste, previsao) ** 0.5,
        "R²": r2_score(y_teste, previsao)
    })

resultados = pd.DataFrame(resultados).sort_values("MAE").reset_index(drop=True)
display(resultados.round(3))

## 7. Interpretação das métricas

O erro absoluto médio (MAE) representa, na unidade do prazo-alvo, o erro absoluto médio das previsões. A raiz do erro quadrático médio (RMSE) também está na unidade do alvo, mas penaliza mais fortemente erros grandes. O coeficiente de determinação (R²) compara a variação explicada pelo modelo com a variação observada, mas não deve ser interpretado isoladamente nem como proporção de previsões corretas.

O modelo de referência estabelece uma linha de base. Um modelo mais complexo só é útil se melhorar o desempenho de forma relevante e estável, sem depender de vazamento de informação. A escolha final também deve considerar custos diferentes de subestimar e superestimar prazos.

In [ ]:
melhor_nome = resultados.loc[0, "Modelo"]
melhor_previsao = predicoes[melhor_nome]

plt.figure(figsize=(7, 6))
plt.scatter(y_teste, melhor_previsao, alpha=0.55)
limite_min = min(y_teste.min(), melhor_previsao.min())
limite_max = max(y_teste.max(), melhor_previsao.max())
plt.plot([limite_min, limite_max], [limite_min, limite_max], "--", color="red")
plt.xlabel("Prazo realizado observado")
plt.ylabel("Prazo previsto")
plt.title(f"Valores observados e previstos: {melhor_nome}")
plt.grid(alpha=0.25)
plt.show()

residuos = y_teste.to_numpy() - melhor_previsao
plt.figure(figsize=(8, 4))
plt.scatter(melhor_previsao, residuos, alpha=0.55)
plt.axhline(0, linestyle="--", color="red")
plt.xlabel("Prazo previsto")
plt.ylabel("Resíduo (observado - previsto)")
plt.title("Resíduos do modelo selecionado")
plt.grid(alpha=0.25)
plt.show()

## 8. Conclusão e limites

A análise ilustra uma tarefa de regressão supervisionada: o modelo aprende a partir de pedidos com prazo realizado conhecido e produz estimativas para pedidos não usados no ajuste. O desempenho deve ser lido em conjunto com a referência, as métricas e os gráficos de diagnóstico.

Os resultados dependem da definição da unidade de análise, da qualidade do prazo-alvo, dos atributos disponíveis no momento da previsão e da estratégia de validação. A divisão aleatória não garante generalização para outros períodos, regiões ou condições operacionais. Os dados Caso Tech têm finalidade didática; os resultados não devem ser tratados como evidência de desempenho operacional sem validação em dados e processos reais.

### Checklist

- Confirmar que cada linha representa um pedido.
- Validar a regra usada para consolidar registros repetidos.
- Confirmar a unidade e o cálculo do prazo realizado.
- Excluir atributos indisponíveis no momento da previsão.
- Comparar os modelos com a referência simples.
- Interpretar MAE e RMSE na unidade do prazo.
- Validar o modelo em dados compatíveis com o cenário de implantação.